In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task
from qc_package.scanner import DbScanner
import pandas as pd

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
TableDetailsModel.objects.filter(table_status=2).count()

508

In [2]:
qc_config = {
    "PATIENT_ID": {"prefix_value": "1001000", "length_of_value": 15},
    "ENCOUNTER_ID": {"prefix_value": "1001000", "length_of_value": 19},
}
db = DbDetailsModel.objects.get(db_name="northwest_metadata")

db_scanner = DbScanner(
    db.source_db_config['connection_str'],
    db.destination_db_config['connection_str'],
    db.run_config["mapping_db_config"],
    qc_config
)

In [4]:
# table_name = 'dl_cube_kpis_op_documents'
# table_obj = TableDetailsModel.objects.get(table_name=table_name)
# table_config = table_obj.table_details_for_ui
# output_result = db_scanner.scan_table(table_name, table_config, table_obj.id)
# output_result

In [5]:
# TableDetailsModel.objects.filter(table_status=2)[126]

In [6]:
# output_result

In [7]:
input_csv = "northwest_phi.csv"
df = pd.read_csv(input_csv)
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

134

In [8]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

377

In [3]:
tables_for_qc = ['additionalnotes','asc_vitals_images_master','assessment_lab_history','assessment_notes_history','assessment_referral_history','assessment_rx_history','assignedpnenc','attachedpastorders','billingdata','case_details','coc','contacts','cpc_measure_config','cpo_cpt_exclude','cpthcpcschanges','cqm_measure_trees','cqm_pcfmeasureconfig','cvxvismapping','derm_lesionusermap','descofservice','diagnosis','dj_ctl_kpi_category','dl_cube_echeckin_data','dl_cube_psms','docsrefdata','doctors','dw_dim_kpi_op','dw_dim_kpis_master','dw_dim_psm','dw_dim_psm_flat_tables','ebo_cptlevel_cascodes','ebo_portal_defaults','ecliniforms','edi_icdcodes','edi_inspayments','edi_inv_cpt','edi_inv_diagnosis','edi_invoice','edi_tos','edi_tos_del','electronichl7content','electronichl7content_archive2','empi_qualityreport','employer','emrereferralattachments','enc','encaddendums','encounterdata','encounters','epcs_reports','eptstmtstemp','examdefaultdetail','formsource','globalkeywords','groupcodes','groupinsurance','hcfa','hl7labnotes','hospitalization','hpi','icdcptmap','ihe_document_bkup','imagingstudies','immforecastingcache','immunizations','inpatientvisit','inscptrules','interactionnotes','itemkeys','items','labattachment_archive','labdata','labdataex','labdataview','laborders','labordersdetails','lsm_actions','measure_compendium','measure_encounterdata_410','measure_encounterdata_430','measure_encounterdata_460','measure_encounterdata_540','measure_encounterdata_560','measure_extract','measure_patientdata_430','measure_provider_configuration','measure_provider_configuration_2018','measure_provider_configuration_2019','measure_provider_configuration_2020','measure_provider_configuration_2021','measure_provider_configuration_2022','measure_provider_configuration_2023','measure_provider_configuration_2024','measure_reportdata_410','measure_reportdata_430','measure_reportdata_460','measure_reportdata_540','measure_reportdata_54024','measure_reportdata_560','measure_reportdata_59024','measure_rxnorm_mapping','medsnotfoundlist','mips_exception_report','mips_practice_settings_measure_conf','mips_submission_2017','ndclookupcrosswalk','ndclookupenteries','nhxpatient','nhxreferral','obhistory','oldrxformularydetails','oldrxmain','opprocedures','ordersets','patientinfo','patients','patients_cca','phm_measures_list','pmorders','pnattachments','prisma_allergy_details','prisma_socialhistory_details','prisma_vital_values','progressnotes','provider_measureconfiguration','provider_measureconfiguration_2017','psac_break_glass','ptinstruction','ptspecificalert','questaoeans','questionnairedata','questionnairemaster','recelectroniclabresults','reconciliation','referral','regsitry_temp_sort','reminderdata','review','rxhub_rxhistory','social','socialhxdistask','statecontroldruglist','structallergyoverride','structdatadetail','structdemographics','structpreventive','structsocialhistory_recover','sunoh_assessment_medical_terms','surescript_changerxoptions','surescript_eprescription','surgicalhistory','surveillance_measures','taxonomycodes','taxonomycodes_del','tblportalcontactsdecrypted','tblwebmsg','trigger_codes_group','userfavoritetests','users','validcpts','visitstatuscodemappings','vitals','vitals_registry','vitalshistory','x12837imapper5010','x12ebs']
len(tables_for_qc)

166

In [4]:
code_failure = ['review']
import traceback
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    if table_obj.table_name in tables_for_qc:
        try:
            table_config = table_obj.table_details_for_ui
            output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
            table_obj.qc_result = output_result
            table_obj.save()
        except Exception as e:
            print(traceback.format_exc())
            code_failure.append(table_obj.table_name)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Traceback (most recent call last):
  File "/var/folders/f8/fl52x_811n33s9_01nh7_b8c0000gn/T/ipykernel_6061/2636900983.py", line 7, in <module>
    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/scanner.py", line 90, in scan_table
    result = detector.is_deidentified(before_rows=source_data, after_rows=sample_data, ignore_condition=table_config.get("ignore_config",{}))
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in is_deidentified
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in <dictcomp>
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
KeyError: 'nd_auto_increment_id'

Traceback (most

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Traceback (most recent call last):
  File "/var/folders/f8/fl52x_811n33s9_01nh7_b8c0000gn/T/ipykernel_6061/2636900983.py", line 7, in <module>
    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/scanner.py", line 90, in scan_table
    result = detector.is_deidentified(before_rows=source_data, after_rows=sample_data, ignore_condition=table_config.get("ignore_config",{}))
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in is_deidentified
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in <dictcomp>
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
KeyError: 'nd_auto_increment_id'



IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



...

Traceback (most recent call last):
  File "/var/folders/f8/fl52x_811n33s9_01nh7_b8c0000gn/T/ipykernel_6061/2636900983.py", line 7, in <module>
    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/scanner.py", line 90, in scan_table
    result = detector.is_deidentified(before_rows=source_data, after_rows=sample_data, ignore_condition=table_config.get("ignore_config",{}))
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in is_deidentified
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in <dictcomp>
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
KeyError: 'nd_auto_increment_id'

Traceback (most

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Traceback (most recent call last):
  File "/var/folders/f8/fl52x_811n33s9_01nh7_b8c0000gn/T/ipykernel_6061/2636900983.py", line 7, in <module>
    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/scanner.py", line 90, in scan_table
    result = detector.is_deidentified(before_rows=source_data, after_rows=sample_data, ignore_condition=table_config.get("ignore_config",{}))
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in is_deidentified
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in <dictcomp>
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
KeyError: 'nd_auto_increment_id'

Traceback (most

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Traceback (most recent call last):
  File "/var/folders/f8/fl52x_811n33s9_01nh7_b8c0000gn/T/ipykernel_6061/2636900983.py", line 7, in <module>
    output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/scanner.py", line 90, in scan_table
    result = detector.is_deidentified(before_rows=source_data, after_rows=sample_data, ignore_condition=table_config.get("ignore_config",{}))
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in is_deidentified
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
  File "/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/qc_package/builders/structured.py", line 91, in <dictcomp>
    before_dict = {row['nd_auto_increment_id']: row for row in before_rows}
KeyError: 'nd_auto_increment_id'

Traceback (most

In [16]:
len(code_failure)
code_failure[0]

'review'

In [12]:
# TableDetailsModel.objects.filter(table_status=2).last().qc_result

In [13]:
import os, json
# table_name, source_row_count, dest_row_count, qc_status, ignore_rows_count, reason, is_phi
all_results = []

def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
file_path = '/NOTEBOOK/Northwest/AUTO_QC_RESULT'
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    if table_obj.table_name in tables_for_qc:
        qc_result = table_obj.qc_result
        if not qc_result:
            continue
        one_table = {
            "table_name": table_obj.table_name,
            "source_rows_count": qc_result['source_rows_count'],
            "dest_rows_count": qc_result['dest_rows_count'],
            "qc_status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
            "ignore_rows_count": qc_result['ignore_rows_count'],
            "reason": qc_result.get("final_qc_result", {}).get("reason"),
            "is_phi": is_phi_table(table_obj.table_details_for_ui)
        }
        all_results.append(one_table)
len(all_results)

88

In [14]:
res_df = pd.DataFrame(all_results)
res_df.shape

(88, 7)

In [15]:
res_df.to_csv("qc_results_05222025.csv", index=False)